# 06 — Demand and performance forecasts with run history

Run the **whole notebook in order**, starting with the settings below.
It reads all saved tracked similarity experiments from 05 by default. For each
experiment it forecasts cumulative Sales, monthly Rental records, and the two
Eilers indices, then saves a separate immutable forecast run. Every experiment
uses the same pinned source snapshots. The final cell displays a batch summary.

## What to set

- `SIMILARITY_EXPERIMENT_IDS = []`: process all tracked experiments currently
  present in the pinned like-games table. Supply a list of 05 experiment IDs to
  process a subset. Alternatively set `SIMILARITY_EXPERIMENT_ID` for one run.
  Experiments overwritten upstream must first be saved again from 05.
- `DISTANCE_CUTOFF`, `MIN_NEIGHBORS`, `FORECAST_MONTHS`, the optional slot-count
  filter, and a writable `MLFLOW_EXPERIMENT_PATH`.
- Source versions may be integers or `None`. `None` pins the latest Delta
  version once during loading. Replay uses those pinned versions.

## Forecast and date rules

| Series | Value | Internal missing months |
| --- | --- | --- |
| Sales | Cumulative sum of monthly `expr1_sum`, revenue `sale`/`sales` | Zero monthly sales, flagged |
| Rental | Monthly `source_row_count` levels | Forward-fill previous monthly level, flagged |
| Eilers | `no_of_slots`-weighted indices by owned/leased population | Forward-fill previous neighbor statistics, flagged |

`prediction_date` is the target's `game_release_date`. A neighbor's monthly row
is eligible only when its month is **strictly before the prediction month**.
This excludes the partially known release month. The first/last available
neighbor months are computed after that filter, so future rows cannot cause
extra historical fills. We do not manufacture months after the last available
neighbor observation. Leading metric nulls remain null.

Month 1 is each source/channel's first observed month. Target calendars use the
target's first observed channel month for plotting, falling back to the release
month when no observations exist. Later target actuals are labels for evaluation,
and never enter the neighbor forecast. `target_curve_starts_before_prediction`
flags inconsistencies between the channel start and the release date; 07 must
define the evaluation policy for them. Missing prediction dates produce no
historical neighbor input, and remain visible as missing forecasts.

These restrictions use observation-month dates. They assume a completed prior
month was available then. The supplied tables do not expose ingestion/publication
dates, so late reporting and revisions cannot be reconstructed from those dates
alone. Pinned Delta versions make this execution reproducible.

## Selection and identity

- Sales, Rental, and Eilers keep their separate lists from 05. Apply the distance
  cutoff, then add nearest candidates until the minimum is reached, subject to
  the saved list size. The minimum is not guaranteed after source availability
  checks. Duplicate matched source names cannot contribute twice.
- 05's P3 parent reservation belongs to its candidate list. The cutoff in this
  notebook may remove that parent. `parent_in_*` fields describe the upstream
  lists; `forecast_neighbors_kept` records the effective lists after filtering.
- `experiment_id` identifies 05's neighbor generation. A fresh setup creates a
  `forecast_run_id`; changing cutoff/horizon creates another run against the
  same neighbor generation. `forecast_mlflow_run_id` links its MLflow run.
- `_target_id` is a stable hash of `target_game_key`; both are retained. Parent
  process and similarity lineage carry into the forecast outputs.
- Use the updated 07 to compare the separately saved runs and select an
  experiment before reviewing a test game.

## Saved outputs

| Table suffix (under `development_042_silver_sandbox.demand_forecast`) | Content |
| --- | --- |
| `demand_forecast_results` | Sales/Rental predictions, actuals, bands, coverage, lineage |
| `performance_forecast_results` | Eilers predictions, raw actuals, display fills, lineage |
| `forecast_neighbors_kept` | Exact retained lists, cutoff/fallback flags, as-of curve availability |
| `forecast_runs` | Completed-run manifest: configuration JSON, source versions, row counts and hashes |

All outputs append. Repeating the batch cell in the same Python session skips
completed experiments and retries failed ones with their original IDs. A fresh
session creates new runs. The completion manifest is written only after all
three result tables have been verified, allowing 07 to ignore partial saves.
Changed settings or snapshots start a new batch. Filter historical tables by
experiment ID and forecast run ID.

The batch summary lists demand/performance row counts and usable prediction
counts. A completed save with zero usable predictions remains visible: missing
historical neighbors or source matches are not hidden as successful coverage.
Notebook 07 evaluates saved completed runs; it does not generate missing forecasts.

MLflow records parameters, coverage, environment versions, and exact Delta
output locations/hashes. Forecast accuracy and selection of the best weights
belong to 07's fixed validation cohort. The ±2 standard-deviation bands describe
neighbor variation; they are not calibrated prediction intervals.

In [0]:
# Edit settings here, then run every remaining cell in order.
like_games_path = "development_042_silver_sandbox.demand_forecast.like_games_ai_attr"
ts_perf_path = "development_042_silver_sandbox.demand_forecast.eilers_group_historical_time_series"
sales_path = "development_042_silver_sandbox.demand_forecast.sales_silver_comp"
lease_path = "development_042_silver_sandbox.demand_forecast.lease_units_bronz"

SIMILARITY_EXPERIMENT_IDS = []  # Empty: ALL tracked experiments in the pinned like-games table.
SIMILARITY_EXPERIMENT_ID = ""   # Optional single-ID shortcut; leave empty for batch mode.
FORECAST_NAME = ""            # Optional readable name.
DISTANCE_CUTOFF = 0.40
MIN_NEIGHBORS = 3
FORECAST_MONTHS = 12
PERFORMANCE_SLOT_COUNTS = []   # Empty: any positive no_of_slots, matching the supplied 06.
LEASE_VALUE_COLUMN = "source_row_count"
LEASE_METRIC_NAME = "monthly_rental_records"
LEASE_CURVE_LABEL = "Monthly rental records"
performance = ["avg_coin_in_index_vs_house", "avg_theo_net_win_index_vs_house"]

SOURCE_TABLE_VERSIONS = {"like_games": None, "sales": None, "lease": None, "performance": None}
MLFLOW_EXPERIMENT_PATH = "/Shared/AI_Feature_Extraction_Forecasts"
IMPLEMENTATION_VERSION = "06_forecasts_v3_batch_mlflow_asof_2026_11_18"
DEMAND_TABLE = "development_042_silver_sandbox.demand_forecast.demand_forecast_results"
PERFORMANCE_TABLE = "development_042_silver_sandbox.demand_forecast.performance_forecast_results"
NEIGHBORS_TABLE = "development_042_silver_sandbox.demand_forecast.forecast_neighbors_kept"
RUNS_TABLE = "development_042_silver_sandbox.demand_forecast.forecast_runs"

In [0]:
import copy
import hashlib
import json
import math
import uuid
from datetime import datetime, timezone
from numbers import Integral, Real
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import MapType


def quoted_table_name(name):
    return ".".join("`" + part.replace("`", "``") + "`" for part in name.split("."))


def normalized_name(column):
    return F.regexp_replace(F.lower(F.trim(column.cast("string"))), r"\s+", " ")


def valid_match_name(column):
    value = normalized_name(column)
    return value.isNotNull() & ~value.isin("", "none", "null", "nan", "na", "n/a", "<na>")


def match_flag(column):
    return F.coalesce(normalized_name(column).isin("true", "1", "yes"), F.lit(False))


def finite_double(column):
    # Source value columns are numeric; discard nonfinite values after casting.
    value = column.cast("double")
    return F.when(value.isNotNull() & ~F.isnan(value) & (F.abs(value) != float("inf")), value)


def require_columns(frame, columns, label):
    missing = sorted(set(columns) - set(frame.columns))
    if missing:
        raise ValueError(f"{label} is missing columns: {missing}")


def assert_unique(frame, keys, label):
    if frame.groupBy(*keys).count().where(F.col("count") > 1).limit(1).count():
        raise ValueError(f"{label} has duplicate keys {keys}; resolve the source rather than choosing a result by distance.")


def canonical_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)


def public_forecast_settings():
    return {
        "similarity_id_requested": SIMILARITY_EXPERIMENT_ID.strip(),
        "similarity_ids_requested": list(SIMILARITY_EXPERIMENT_IDS),
        "source_paths": {"like_games": like_games_path, "sales": sales_path,
                         "lease": lease_path, "performance": ts_perf_path},
        "source_versions_requested": copy.deepcopy(SOURCE_TABLE_VERSIONS),
        "distance_cutoff": float(DISTANCE_CUTOFF), "minimum_neighbors": int(MIN_NEIGHBORS),
        "horizon_months": int(FORECAST_MONTHS), "performance_metrics": list(performance),
        "slot_counts": sorted(float(x) for x in PERFORMANCE_SLOT_COUNTS),
        "lease_value_column": LEASE_VALUE_COLUMN, "lease_metric_name": LEASE_METRIC_NAME,
        "lease_curve_label": LEASE_CURVE_LABEL, "forecast_name": FORECAST_NAME,
        "mlflow_experiment_path": MLFLOW_EXPERIMENT_PATH, "implementation_version": IMPLEMENTATION_VERSION,
        "outputs": {"demand": DEMAND_TABLE, "performance": PERFORMANCE_TABLE,
                    "neighbors": NEIGHBORS_TABLE, "runs": RUNS_TABLE},
    }


if (isinstance(DISTANCE_CUTOFF, bool) or not isinstance(DISTANCE_CUTOFF, Real)
    or not math.isfinite(float(DISTANCE_CUTOFF)) or DISTANCE_CUTOFF < 0):
    raise ValueError("DISTANCE_CUTOFF must be a finite nonnegative number.")
for label, value, minimum in [("MIN_NEIGHBORS", MIN_NEIGHBORS, 0), ("FORECAST_MONTHS", FORECAST_MONTHS, 1)]:
    if isinstance(value, bool) or not isinstance(value, Integral) or value < minimum:
        raise ValueError(f"{label} must be an integer >= {minimum}.")
if any(isinstance(x, bool) or not isinstance(x, Real) or not math.isfinite(float(x)) or x <= 0
       for x in PERFORMANCE_SLOT_COUNTS):
    raise ValueError("PERFORMANCE_SLOT_COUNTS must contain positive finite numbers.")
if len(PERFORMANCE_SLOT_COUNTS) != len(set(PERFORMANCE_SLOT_COUNTS)):
    raise ValueError("PERFORMANCE_SLOT_COUNTS contains duplicates.")
if not performance or len(performance) != len(set(performance)):
    raise ValueError("Provide a nonempty list of distinct performance metrics.")
if not MLFLOW_EXPERIMENT_PATH.startswith("/"):
    raise ValueError("Use an absolute MLflow workspace experiment path.")

LOADED_SETTINGS = public_forecast_settings()
if set(SOURCE_TABLE_VERSIONS) != set(LOADED_SETTINGS["source_paths"]):
    raise ValueError("SOURCE_TABLE_VERSIONS must contain like_games, sales, lease, performance.")
RESOLVED_SOURCE_VERSIONS = {}
source_frames = {}
for label, path in LOADED_SETTINGS["source_paths"].items():
    requested = SOURCE_TABLE_VERSIONS[label]
    if requested is not None and (isinstance(requested, bool) or not isinstance(requested, Integral) or requested < 0):
        raise ValueError(f"Invalid version for {label}: {requested!r}")
    version = int(requested) if requested is not None else int(
        spark.sql(f"DESCRIBE HISTORY {quoted_table_name(path)}")
        .agg(F.max("version").alias("v")).first()["v"]
    )
    RESOLVED_SOURCE_VERSIONS[label] = version
    source_frames[label] = spark.read.option("versionAsOf", version).table(path)
    print(f"Pinned {label}: {path}, Delta version {version}")

all_like_games_df = source_frames["like_games"]
sales_df, lease_df, ts_perf_df = (source_frames[x] for x in ["sales", "lease", "performance"])
require_columns(all_like_games_df, {
    "join_game_name", "experiment_id", "experiment_readable_name", "game_release_date",
    "sales_similar_games", "rental_similar_games", "eilers_similar_games",
    "similarity_mlflow_run_id", "similarity_config_hash", "parent_game_process",
    "distance_method", "feature_group_weights", "own_status",
    "has_sales_match", "has_rental_match", "has_eilers_ts_match",
    "to_sales_name_match", "to_rental_name_match", "to_eilers_ts_name_match",
}, "like-games table: run updated notebook 05 first")
available_experiments_df = (
    all_like_games_df.where(F.col("similarity_mlflow_run_id").isNotNull())
    .select("experiment_id", "experiment_readable_name", "parent_game_process", "distance_method")
    .distinct()
)
# Resolve the full batch before producing any outputs. Every experiment uses
# the same pinned commercial/performance snapshots for fair evaluation in 07.
if not isinstance(SIMILARITY_EXPERIMENT_IDS, list) or any(
    not isinstance(value, str) or not value.strip() for value in SIMILARITY_EXPERIMENT_IDS
):
    raise ValueError("SIMILARITY_EXPERIMENT_IDS must be a list of nonempty IDs.")
requested_ids = [value.strip() for value in SIMILARITY_EXPERIMENT_IDS]
if len(requested_ids) != len(set(requested_ids)):
    raise ValueError("SIMILARITY_EXPERIMENT_IDS contains duplicates.")
if SIMILARITY_EXPERIMENT_ID.strip() and requested_ids:
    raise ValueError("Use SIMILARITY_EXPERIMENT_ID or SIMILARITY_EXPERIMENT_IDS, not both.")
available_ids = sorted(row["experiment_id"] for row in
    available_experiments_df.select("experiment_id").distinct().collect()
    if row["experiment_id"] is not None)
SELECTED_SIMILARITY_EXPERIMENT_IDS = (
    [SIMILARITY_EXPERIMENT_ID.strip()] if SIMILARITY_EXPERIMENT_ID.strip()
    else requested_ids or available_ids
)
unknown_ids = sorted(set(SELECTED_SIMILARITY_EXPERIMENT_IDS) - set(available_ids))
if unknown_ids:
    raise ValueError(f"Unknown/untracked similarity experiment IDs: {unknown_ids}")
if not SELECTED_SIMILARITY_EXPERIMENT_IDS:
    raise ValueError("No tracked experiments found in the pinned like-games table. Save them with 05 first.")
for experiment_id in SELECTED_SIMILARITY_EXPERIMENT_IDS:
    identities = (all_like_games_df.where(F.col("experiment_id") == experiment_id)
                  .select("similarity_mlflow_run_id", "similarity_config_hash").distinct().collect())
    if len(identities) != 1 or not all(identities[0][name] for name in
                                      ["similarity_mlflow_run_id", "similarity_config_hash"]):
        raise ValueError(f"Experiment {experiment_id} does not identify one complete tracked generation from 05.")
print(f"Selected {len(SELECTED_SIMILARITY_EXPERIMENT_IDS)} similarity experiments; each will get its own demand AND performance run.")
display(available_experiments_df.where(F.col("experiment_id").isin(SELECTED_SIMILARITY_EXPERIMENT_IDS)))


In [0]:
# Forecast, validate, and save ONE experiment in an isolated local scope.
# The batch cell below calls this same pipeline for each selected experiment.
def run_similarity_forecast(experiment_id, run_state):
    SELECTED_SIMILARITY_EXPERIMENT_ID = experiment_id
    OUTPUT_FRAMES = {}
    FORECAST_MLFLOW_RUN_ID = run_state.get("forecast_mlflow_run_id")
    SAVE_STATE = run_state.get("save_identity")
    try:
        like_games_df = all_like_games_df.where(F.col("experiment_id") == SELECTED_SIMILARITY_EXPERIMENT_ID)
        identities = like_games_df.select("similarity_mlflow_run_id", "similarity_config_hash").distinct().collect()
        if len(identities) != 1 or not all(identities[0][name] for name in ["similarity_mlflow_run_id", "similarity_config_hash"]):
            raise ValueError("Select one complete tracked similarity generation from updated 05.")
        SIMILARITY_IDENTITY = identities[0].asDict()
        similarity_metadata = like_games_df.select(
            "distance_method", "feature_group_weights", "parent_game_process"
        ).first().asDict()
        print(f"Selected similarity experiment: {SELECTED_SIMILARITY_EXPERIMENT_ID}")
        display(available_experiments_df.where(F.col("experiment_id") == SELECTED_SIMILARITY_EXPERIMENT_ID))

        # One immutable configuration and a fresh forecast ID for this execution.
        if public_forecast_settings() != LOADED_SETTINGS:
            raise ValueError("Settings changed after loading. Rerun loading before setup.")

        FORECAST_CONFIG = {
            **copy.deepcopy(LOADED_SETTINGS),
            "similarity_experiment_id": SELECTED_SIMILARITY_EXPERIMENT_ID,
            **SIMILARITY_IDENTITY,
            "distance_method": similarity_metadata["distance_method"],
            "feature_group_weights": {
                group: float(similarity_metadata["feature_group_weights"].get(group, 0.0))
                for group in ["themes", "mechanics", "reels", "ways", "persistence", "months_since_release"]
            },
            "parent_game_process": similarity_metadata["parent_game_process"],
            "source_versions_resolved": copy.deepcopy(RESOLVED_SOURCE_VERSIONS),
            "prediction_date_rule": "target.game_release_date",
            "neighbor_observation_rule": "source_month < first_day_of_target_prediction_month",
            "reporting_assumption": "completed prior month available; no ingestion/publication timestamp supplied",
            "calendar_rule": "relative to first observed source month; target labels use first channel observation",
            "sales_fill": "internal missing monthly rows are zero; then cumulative sum",
            "lease_fill": "internal missing monthly levels forward-filled",
            "performance_fill": "internal missing neighbor metric statistics forward-filled; leading nulls remain",
            "fill_bounds": "stop at last source observation available as of prediction month",
            "performance_weighting": "no_of_slots; per-metric nonnull denominator; forecast duplicates averaged per slot bucket",
            "target_actual_policy": "full pinned source snapshot; imputation flagged; score raw observed values in 07",
            "bands": "population standard deviation across contributing values; mean +/- 2 std",
            "neighbor_selection": "exclude target source mapping, deduplicate normalized source names, cutoff then nearest minimum fallback",
            "p3_parent_after_cutoff": "normal cutoff/fallback retention; upstream parent reservation may be removed",
            "stable_target_id": "Spark xxhash64(normalized canonical game key)",
        }
        FORECAST_CONFIG_JSON = canonical_json(FORECAST_CONFIG)
        FORECAST_CONFIG_HASH = hashlib.sha256(FORECAST_CONFIG_JSON.encode()).hexdigest()
        FORECAST_RUN_ID = run_state.setdefault("forecast_run_id", "fcst_" + uuid.uuid4().hex)
        FORECAST_MLFLOW_RUN_ID = run_state.get("forecast_mlflow_run_id")
        FORECAST_RUN_STARTED_AT_UTC = run_state.setdefault("started_at_utc", datetime.now(timezone.utc).isoformat())
        SAVE_STATE = run_state.get("save_identity")


        def assert_forecast_settings():
            if public_forecast_settings() != LOADED_SETTINGS or canonical_json(FORECAST_CONFIG) != FORECAST_CONFIG_JSON:
                raise ValueError("Configuration changed. Rerun loading/setup and downstream cells for a new forecast run.")


        # New metadata is additive; optional original descriptors may be absent in legacy data.
        optional_strings = ["parent_game", "parent_game_match_status", "resolved_parent_game_name",
                            "game_category", "parent_own_status", "game_classification"]
        for name in optional_strings:
            if name not in like_games_df.columns:
                like_games_df = like_games_df.withColumn(name, F.lit(None).cast("string"))
        for name in ["parent_in_sales_neighbors", "parent_in_rental_neighbors"]:
            if name not in like_games_df.columns:
                like_games_df = like_games_df.withColumn(name, F.lit(None).cast("boolean"))

        sales_target_eligible = match_flag(F.col("has_sales_match")) & valid_match_name(F.col("to_sales_name_match"))
        rental_target_eligible = match_flag(F.col("has_rental_match")) & valid_match_name(F.col("to_rental_name_match"))
        like_games_source_df = (
            like_games_df
            .where(valid_match_name(F.col("join_game_name")))
            .where(match_flag(F.col("has_eilers_ts_match")) & (sales_target_eligible | rental_target_eligible))
            .withColumn("target_game_key", normalized_name(F.col("join_game_name")))
            .withColumn("_target_id", F.xxhash64("target_game_key"))
            .withColumn("target_game", F.col("join_game_name"))
            .withColumn("game_release_date", F.to_date("game_release_date"))
            .withColumn("prediction_date", F.col("game_release_date"))
            .withColumn("experiment_name", F.coalesce("experiment_readable_name", "experiment_id"))
            .withColumn("forecast_run_id", F.lit(FORECAST_RUN_ID))
            .withColumn("forecast_config_hash", F.lit(FORECAST_CONFIG_HASH))
        )
        assert_unique(like_games_source_df, ["target_game_key"], "Selected similarity generation")
        assert_unique(like_games_source_df, ["_target_id"], "Stable target IDs")
        if not like_games_source_df.limit(1).count():
            raise ValueError("No usable target mappings in the selected similarity generation.")

        TARGET_METADATA = [
            "_target_id", "target_game", "target_game_key", "game_release_date", "prediction_date",
            "experiment_id", "experiment_name", "distance_method", "feature_group_weights",
            "similarity_mlflow_run_id", "similarity_config_hash", "forecast_run_id", "forecast_config_hash",
            "has_sales_match", "has_rental_match", "has_eilers_ts_match",
            "to_sales_name_match", "to_rental_name_match", "to_eilers_ts_name_match",
            "parent_game", "parent_game_process", "parent_game_match_status", "resolved_parent_game_name",
            "parent_in_sales_neighbors", "parent_in_rental_neighbors",
            "game_category", "parent_own_status", "game_classification",
        ]
        target_info_df = like_games_source_df.select(*TARGET_METADATA, "own_status")
        target_channel_info_df = (
            like_games_source_df
            .withColumn("revenue_type", F.explode(F.array(
                F.when(sales_target_eligible, F.lit("sales")),
                F.when(rental_target_eligible, F.lit("lease")),
            )))
            .where(F.col("revenue_type").isNotNull())
            .select(*TARGET_METADATA, "revenue_type", F.col("own_status").alias("target_own_status"))
            .withColumn("game_id", F.col("_target_id"))
            .withColumn("own_status", F.when(F.col("revenue_type") == "sales", "owned").otherwise("leased"))
            .withColumn("metric_name", F.when(F.col("revenue_type") == "sales", "cumulative_sales").otherwise(LEASE_METRIC_NAME))
            .withColumn("curve_label", F.when(F.col("revenue_type") == "sales", "Cumulative sales").otherwise(LEASE_CURVE_LABEL))
            .withColumn("curve_kind", F.when(F.col("revenue_type") == "sales", "cumulative").otherwise("monthly"))
            .withColumn("_source_key", normalized_name(F.when(
                F.col("revenue_type") == "sales", F.col("to_sales_name_match")
            ).otherwise(F.col("to_rental_name_match"))))
        )
        print(f"Forecast run: {FORECAST_RUN_ID}")
        print(f"Configuration hash: {FORECAST_CONFIG_HASH}")

        # Apply cutoff/fallback separately to each source. Select unique source mappings.
        assert_forecast_settings()


        def keep_neighbors_with_minimum(source_df, list_column, neighbor_type):
            target_mapping_column = {
                "sales": "to_sales_name_match", "rental": "to_rental_name_match", "eilers": "to_eilers_ts_name_match"
            }[neighbor_type]
            target_mapping = F.col(target_mapping_column)
            if neighbor_type == "eilers":
                target_mapping = F.when(valid_match_name(target_mapping), target_mapping).otherwise(F.col("target_game"))
            target_mappings = source_df.select(
                "_target_id", normalized_name(target_mapping).alias("_target_source_key")
            )
            candidates = (
                source_df.select("_target_id", F.explode(F.col(list_column)).alias("_neighbor"))
                .select("_target_id",
                        F.col("_neighbor.join_game_name_key").alias("neighbor_game"),
                        F.col("_neighbor.matched_source_name").alias("matched_source_name"),
                        finite_double(F.col("_neighbor.distance")).alias("distance"),
                        finite_double(F.col("_neighbor.similarity_score")).alias("similarity_score"))
                .where(valid_match_name(F.col("matched_source_name")) & F.col("distance").isNotNull()
                       & (F.col("distance") >= 0))
                .withColumn("_neighbor_source_key", normalized_name(F.col("matched_source_name")))
                .join(target_mappings, "_target_id", "inner")
                .where(~F.col("_neighbor_source_key").eqNullSafe(F.col("_target_source_key")))
            )
            source_order = Window.partitionBy("_target_id", "_neighbor_source_key").orderBy(
                F.col("distance").asc(), F.col("neighbor_game").asc_nulls_last(), F.col("matched_source_name").asc())
            candidates = candidates.withColumn("_source_rank", F.row_number().over(source_order)).where(F.col("_source_rank") == 1)
            rank_window = Window.partitionBy("_target_id").orderBy(
                F.col("distance").asc(), F.col("neighbor_game").asc_nulls_last(), F.col("_neighbor_source_key").asc())
            retained = (
                candidates.withColumn("neighbor_rank", F.row_number().over(rank_window))
                .where((F.col("distance") <= DISTANCE_CUTOFF) | (F.col("neighbor_rank") <= MIN_NEIGHBORS))
                .withColumn("neighbor_type", F.lit(neighbor_type))
                .withColumn("retained_within_cutoff", F.col("distance") <= DISTANCE_CUTOFF)
                .withColumn("retained_as_min_neighbor_fallback", F.col("distance") > DISTANCE_CUTOFF)
                .drop("_source_rank", "_target_source_key")
                .join(target_info_df, "_target_id", "inner")
            )
            return retained


        sales_neighbors_kept_df = keep_neighbors_with_minimum(
            like_games_source_df.where(sales_target_eligible), "sales_similar_games", "sales")
        rental_neighbors_kept_df = keep_neighbors_with_minimum(
            like_games_source_df.where(rental_target_eligible), "rental_similar_games", "rental")
        eilers_neighbors_kept_df = keep_neighbors_with_minimum(
            like_games_source_df, "eilers_similar_games", "eilers")
        all_neighbors_kept_df = (
            sales_neighbors_kept_df.unionByName(rental_neighbors_kept_df).unionByName(eilers_neighbors_kept_df)
        )
        NEIGHBOR_SELECTION_RUN_ID = FORECAST_RUN_ID
        display(all_neighbors_kept_df.groupBy("neighbor_type").agg(
            F.countDistinct("target_game_key").alias("targets_with_retained_neighbors"),
            F.count("neighbor_game").alias("retained_neighbors"),
            F.avg("distance").alias("mean_distance_within_this_configuration"),
        ))

        assert_forecast_settings()
        # Commercial curves: cumulative sales and monthly rental records.
        # Neighbor observations are cut off before alignment or gap filling.

        require_columns(
            sales_df,
            {"normalized_game_name", "revenue_type", "beginning_month_date", "expr1_sum"},
            "Sales source",
        )
        require_columns(
            lease_df, {"theme_name_eilers", "month", LEASE_VALUE_COLUMN}, "Rental source"
        )

        sales_monthly_source_df = (
            sales_df
            .where(normalized_name(F.col("revenue_type")).isin("sale", "sales"))
            .where(valid_match_name(F.col("normalized_game_name")))
            .select(
                normalized_name(F.col("normalized_game_name")).alias("_source_key"),
                F.lit("sales").alias("revenue_type"),
                F.trunc(F.to_date("beginning_month_date"), "month").alias("_calendar_month"),
                finite_double(F.col("expr1_sum")).alias("_monthly_value"),
            )
        )
        lease_monthly_source_df = (
            lease_df
            .where(valid_match_name(F.col("theme_name_eilers")))
            .select(
                normalized_name(F.col("theme_name_eilers")).alias("_source_key"),
                F.lit("lease").alias("revenue_type"),
                F.trunc(F.to_date("month"), "month").alias("_calendar_month"),
                finite_double(F.col(LEASE_VALUE_COLUMN)).alias("_monthly_value"),
            )
        )
        source_keys = ["revenue_type", "_source_key"]
        source_monthly_df = (
            sales_monthly_source_df.unionByName(lease_monthly_source_df)
            .where(F.col("_calendar_month").isNotNull() & F.col("_monthly_value").isNotNull())
            .groupBy(*source_keys, "_calendar_month")
            .agg(F.sum("_monthly_value").alias("_monthly_value"))
        )


        def commercial_curves(monthly_df, curve_keys):
            """Align observed months, filling internal gaps within known source coverage.

            Callers pass either the full snapshot for target actuals or observations
            already filtered to the target prediction date for neighbor inputs.
            """
            bounds_df = monthly_df.groupBy(*curve_keys).agg(
                F.min("_calendar_month").alias("curve_start_month"),
                F.max("_calendar_month").alias("last_observed_source_month"),
                F.count("_monthly_value").alias("source_months_observed"),
            )
            bounds_df = bounds_df.withColumn(
                "_last_month",
                F.least(
                    F.floor(F.months_between("last_observed_source_month", "curve_start_month"))
                    .cast("int") + 1,
                    F.lit(FORECAST_MONTHS),
                ),
            )
            scaffold_df = (
                bounds_df
                .withColumn("month_index", F.explode(F.sequence(F.lit(1), F.col("_last_month"))))
                .withColumn("_calendar_month", F.add_months("curve_start_month", F.col("month_index") - 1))
                .drop("_last_month")
            )
            fill_window = (
                Window.partitionBy(*curve_keys).orderBy("month_index")
                .rowsBetween(Window.unboundedPreceding, Window.currentRow)
            )
            curves_df = (
                scaffold_df
                .join(monthly_df.select(*curve_keys, "_calendar_month", "_monthly_value"),
                      on=curve_keys + ["_calendar_month"], how="left")
                .withColumn("actual_is_imputed", F.col("_monthly_value").isNull())
                .withColumn("actual_is_observed", F.col("_monthly_value").isNotNull())
                .withColumn(
                    "_filled_monthly_value",
                    F.when(F.col("revenue_type") == "sales", F.coalesce("_monthly_value", F.lit(0.0)))
                    .otherwise(F.last("_monthly_value", ignorenulls=True).over(fill_window)),
                )
                .withColumn(
                    "source_curve_value",
                    F.when(F.col("revenue_type") == "sales", F.sum("_filled_monthly_value").over(fill_window))
                    .otherwise(F.col("_filled_monthly_value")),
                )
                .withColumn(
                    "actual_contains_imputed_history",
                    F.when(
                        F.col("revenue_type") == "sales",
                        F.max(F.col("actual_is_imputed").cast("int")).over(fill_window) > 0,
                    ).otherwise(F.col("actual_is_imputed")),
                )
            )
            return curves_df


        # Full, pinned source history is used only to attach target actuals.
        source_curves_df = commercial_curves(source_monthly_df, source_keys)
        source_bounds_df = source_monthly_df.groupBy(*source_keys).agg(
            F.min("_calendar_month").alias("actual_curve_start_month")
        )

        # A source mapping contributes once, even when several AI game names map to it.
        neighbor_columns = ["_target_id", "neighbor_game", "matched_source_name", "distance"]
        demand_neighbors_kept_df = (
            sales_neighbors_kept_df.select(*neighbor_columns)
            .withColumn("revenue_type", F.lit("sales"))
            .unionByName(
                rental_neighbors_kept_df.select(*neighbor_columns)
                .withColumn("revenue_type", F.lit("lease"))
            )
            .withColumn("_source_key", normalized_name(F.col("matched_source_name")))
        )
        source_neighbor_rank = Window.partitionBy("_target_id", "revenue_type", "_source_key").orderBy(
            F.col("distance").asc_nulls_last(),
            F.col("neighbor_game").asc_nulls_last(),
            F.col("matched_source_name").asc_nulls_last(),
        )
        demand_neighbors_kept_df = (
            demand_neighbors_kept_df
            .where(valid_match_name(F.col("matched_source_name")))
            .withColumn("_source_rank", F.row_number().over(source_neighbor_rank))
            .where(F.col("_source_rank") == 1)
            .drop("_source_rank")
            .join(
                target_channel_info_df.select(
                    "_target_id", "revenue_type", "target_game", "target_game_key",
                    "prediction_date", "own_status", "forecast_run_id",
                ),
                on=["_target_id", "revenue_type"], how="inner",
            )
        )

        # Completed monthly observations only. This happens BEFORE first/last month
        # discovery, scaffolding and filling, so later source rows cannot extend a curve.
        neighbor_monthly_asof_df = (
            demand_neighbors_kept_df
            .join(source_monthly_df, on=source_keys, how="inner")
            .where(F.col("_calendar_month") < F.trunc("prediction_date", "month"))
        )
        neighbor_curve_keys = ["_target_id", "revenue_type", "_source_key"]
        neighbor_demand_curves_df = (
            commercial_curves(
                neighbor_monthly_asof_df.select(*neighbor_curve_keys, "_calendar_month", "_monthly_value"),
                neighbor_curve_keys,
            )
            .join(demand_neighbors_kept_df, on=neighbor_curve_keys, how="inner")
            .withColumn("neighbor_type", F.when(F.col("revenue_type") == "sales", "sales").otherwise("rental"))
        )

        demand_baseline_df = (
            neighbor_demand_curves_df
            .groupBy("_target_id", "revenue_type", "month_index")
            .agg(
                F.avg("source_curve_value").alias("forecast_value"),
                F.stddev_pop("source_curve_value").alias("_stddev"),
                F.countDistinct("_source_key").alias("neighbor_count"),
                F.sum(F.col("actual_is_imputed").cast("long")).alias("filled_neighbor_count"),
                F.sum(F.col("actual_contains_imputed_history").cast("long"))
                .alias("neighbors_with_imputed_history"),
            )
            .withColumn("_stddev", F.coalesce("_stddev", F.lit(0.0)))
            .withColumn("observed_neighbor_count", F.col("neighbor_count") - F.col("filled_neighbor_count"))
            .withColumn("high_band_2std", F.col("forecast_value") + 2 * F.col("_stddev"))
            .withColumn("low_band_2std", F.greatest(F.col("forecast_value") - 2 * F.col("_stddev"), F.lit(0.0)))
            .drop("_stddev")
        )

        # Targets survive even when there is no usable neighbor history or actual.
        # Curve month 1 remains the target channel's oldest observed month. When a
        # target has no observations, its prediction month supplies the plotting date.
        demand_forecast_curve_df = (
            target_channel_info_df
            .join(source_bounds_df, on=source_keys, how="left")
            .withColumn("curve_start_is_inferred", F.col("actual_curve_start_month").isNull())
            .withColumn("curve_start_month", F.coalesce("actual_curve_start_month", F.trunc("prediction_date", "month")))
            .withColumn(
                "target_curve_starts_before_prediction",
                F.coalesce(F.col("actual_curve_start_month") < F.trunc("prediction_date", "month"), F.lit(False)),
            )
            .withColumn("month_index", F.explode(F.sequence(F.lit(1), F.lit(FORECAST_MONTHS))))
            .join(demand_baseline_df, on=["_target_id", "revenue_type", "month_index"], how="left")
            .join(
                source_curves_df.select(
                    *source_keys, "month_index", F.col("source_curve_value").alias("actual_value"),
                    "actual_is_imputed", "actual_is_observed", "actual_contains_imputed_history",
                    F.col("_monthly_value").alias("actual_monthly_observed_value"),
                ),
                on=source_keys + ["month_index"], how="left",
            )
            .withColumn("actual_is_observed", F.coalesce("actual_is_observed", F.lit(False)))
            .withColumn("forecast_month", F.add_months("curve_start_month", F.col("month_index") - 1))
            .fillna({"neighbor_count": 0, "filled_neighbor_count": 0,
                     "observed_neighbor_count": 0, "neighbors_with_imputed_history": 0})
            .withColumn("baseline_mean_sales", F.when(F.col("revenue_type") == "sales", F.col("forecast_value")))
            .withColumn("actual_cumulative_sales", F.when(F.col("revenue_type") == "sales", F.col("actual_value")))
            .withColumn("actual_sales_month_zero_filled", F.when(F.col("revenue_type") == "sales", F.col("actual_is_imputed")))
            .withColumn("baseline_mean_lease", F.when(F.col("revenue_type") == "lease", F.col("forecast_value")))
            .withColumn("actual_monthly_lease", F.when(F.col("revenue_type") == "lease", F.col("actual_value")))
            .drop("_source_key")
        )

        COMMERCIAL_RUN_ID = FORECAST_RUN_ID
        display(demand_forecast_curve_df.orderBy("target_game", "revenue_type", "month_index"))

        assert_forecast_settings()
        # Eilers forecasts: the neighbor history is restricted to completed months
        # before each target's prediction month. Target actuals use the pinned source
        # snapshot only for evaluation and calendar labels.

        require_columns(
            ts_perf_df,
            {"game_name", "own_status", "yearmonth", "no_of_slots", *performance},
            "Eilers performance source",
        )


        def yearmonth_to_date(column):
            return F.trunc(F.to_date(column.cast("string"), "yyyy-MM-dd"), "month")


        performance_source_df = (
            ts_perf_df
            .withColumn("own_status", normalized_name(F.col("own_status")))
            .withColumn("_game_key", normalized_name(F.col("game_name")))
            .withColumn("_perf_month", yearmonth_to_date(F.col("yearmonth")))
            .withColumn("no_of_slots", finite_double(F.col("no_of_slots")))
            .where(
                F.col("own_status").isin("owned", "leased")
                & valid_match_name(F.col("game_name"))
                & (F.col("no_of_slots") > 0)
                & F.col("_perf_month").isNotNull()
            )
        )
        if PERFORMANCE_SLOT_COUNTS:
            performance_source_df = performance_source_df.where(
                F.col("no_of_slots").isin([float(value) for value in PERFORMANCE_SLOT_COUNTS])
            )
        for metric in performance:
            performance_source_df = performance_source_df.withColumn(
                metric, finite_double(F.col(metric))
            )

        # Calendar anchors may use observed target dates; neighbor forecasts never
        # consume target actual values. Missing target ownership histories stay present.
        target_base_df = (
            target_channel_info_df
            .drop("_source_key", "metric_name", "curve_label", "curve_kind")
            .withColumn(
                "_game_key",
                normalized_name(F.when(
                    valid_match_name(F.col("to_eilers_ts_name_match")),
                    F.col("to_eilers_ts_name_match"),
                ).otherwise(F.col("target_game"))),
            )
        )
        target_performance_bounds_df = (
            performance_source_df.groupBy("_game_key", "own_status")
            .agg(
                F.min("_perf_month").alias("_target_start_month"),
                F.max("_perf_month").alias("_target_perf_last_month"),
            )
        )
        performance_targets_df = (
            target_base_df
            .join(target_performance_bounds_df, on=["_game_key", "own_status"], how="left")
            .withColumn("target_first_observed_month", F.col("_target_start_month"))
            .withColumn("target_start_is_observed", F.col("_target_start_month").isNotNull())
            .withColumn(
                "target_curve_starts_before_prediction",
                F.col("_target_start_month") < F.trunc(F.col("prediction_date"), "month"),
            )
            .withColumn(
                "curve_start_month",
                F.coalesce("_target_start_month", F.trunc(F.col("prediction_date"), "month")),
            )
        )

        # One canonical mapping contributes once, even when multiple AI games refer
        # to the same Eilers name. Ownership comes from the target's selected channel.
        performance_neighbor_rank = Window.partitionBy(
            "_target_id", "_neighbor_key"
        ).orderBy(F.col("distance").asc(), F.col("neighbor_game").asc())
        performance_neighbor_keys_df = (
            eilers_neighbors_kept_df
            .select("_target_id", "neighbor_game", "matched_source_name", "distance", "similarity_score")
            .withColumn("_neighbor_key", normalized_name(F.col("matched_source_name")))
            .where(valid_match_name(F.col("_neighbor_key")))
            .withColumn("_mapping_rank", F.row_number().over(performance_neighbor_rank))
            .where(F.col("_mapping_rank") == 1)
            .drop("_mapping_rank")
        )
        performance_neighbor_targets_df = (
            performance_neighbor_keys_df
            .join(
                performance_targets_df.select(
                    "_target_id", "target_game", "target_game_key", "own_status", "revenue_type",
                    "prediction_date", "forecast_run_id", "forecast_config_hash",
                ),
                on="_target_id", how="inner",
            )
        )
        n = performance_neighbor_targets_df.alias("n")
        p = performance_source_df.alias("p")
        neighbor_perf_df = (
            n.join(
                p,
                (F.col("n._neighbor_key") == F.col("p._game_key"))
                & (F.col("n.own_status") == F.col("p.own_status"))
                # Filter before deriving first/last dates, relative months or fills.
                & (F.col("p._perf_month") < F.trunc(F.col("n.prediction_date"), "month")),
                "inner",
            )
            .select(
                "n.*", F.col("p.no_of_slots"), F.col("p._perf_month"),
                *[F.col(f"p.{metric}").alias(metric) for metric in performance],
            )
        )
        neighbor_window = Window.partitionBy("_target_id", "own_status", "_neighbor_key")
        neighbor_perf_df = (
            neighbor_perf_df
            .withColumn("_neighbor_start_month", F.min("_perf_month").over(neighbor_window))
            .withColumn(
                "month_index",
                F.floor(F.months_between("_perf_month", "_neighbor_start_month")).cast("int") + 1,
            )
        )

        # Preserve the existing forecast duplicate policy: average rows at the same
        # neighbor/month/slot count, then use that slot count as the aggregation weight.
        neighbor_game_keys = ["_target_id", "target_game", "own_status", "_neighbor_key"]
        neighbor_monthly_df = (
            neighbor_perf_df.where(F.col("month_index").between(1, FORECAST_MONTHS))
            .groupBy(*neighbor_game_keys, "no_of_slots", "month_index")
            .agg(*[F.avg(metric).alias(metric) for metric in performance])
        )
        aggregations = []
        for metric in performance:
            valid_weight = F.when(F.col(metric).isNotNull(), F.col("no_of_slots"))
            aggregations.extend([
                F.sum(valid_weight).alias(f"_{metric}_weight_sum"),
                F.sum(valid_weight * F.col(metric)).alias(f"_{metric}_weighted_sum"),
                F.sum(valid_weight * F.pow(F.col(metric), 2)).alias(f"_{metric}_weighted_sq_sum"),
            ])
        neighbor_monthly_stats_df = (
            neighbor_monthly_df.groupBy(*neighbor_game_keys, "month_index").agg(*aggregations)
        )

        # Last coverage is computed from AS-OF observations, including records beyond
        # the plotted horizon. A future observation cannot justify filling a past gap.
        neighbor_month_ranges_df = (
            neighbor_perf_df.groupBy(*neighbor_game_keys)
            .agg(F.least(F.max("month_index"), F.lit(FORECAST_MONTHS)).alias("_last_month"))
        )
        neighbor_month_scaffold_df = (
            neighbor_month_ranges_df
            .withColumn("month_index", F.explode(F.sequence(F.lit(1), F.col("_last_month"))))
            .drop("_last_month")
        )
        neighbor_aligned_stats_df = neighbor_month_scaffold_df.join(
            neighbor_monthly_stats_df, on=neighbor_game_keys + ["month_index"], how="left"
        )
        neighbor_fill_window = (
            Window.partitionBy(*neighbor_game_keys).orderBy("month_index")
            .rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
        filled_aggregations = []
        for metric in performance:
            stat_columns = [
                f"_{metric}_weight_sum", f"_{metric}_weighted_sum", f"_{metric}_weighted_sq_sum"
            ]
            observed_col = f"{metric}_is_observed"
            filled_col = f"{metric}_is_filled"
            valid_stats = F.when(
                F.col(stat_columns[0]) > 0,
                F.struct(*[F.col(name).alias(name) for name in stat_columns]),
            )
            neighbor_aligned_stats_df = (
                neighbor_aligned_stats_df
                .withColumn(observed_col, F.coalesce(F.col(stat_columns[0]) > 0, F.lit(False)))
                .withColumn("_last_metric_stats", F.last(valid_stats, ignorenulls=True).over(neighbor_fill_window))
                .withColumn(filled_col, (~F.col(observed_col)) & F.col("_last_metric_stats").isNotNull())
            )
            for name in stat_columns:
                neighbor_aligned_stats_df = neighbor_aligned_stats_df.withColumn(
                    name, F.col("_last_metric_stats").getField(name)
                )
            neighbor_aligned_stats_df = neighbor_aligned_stats_df.drop("_last_metric_stats")
            filled_aggregations.extend([
                *[F.sum(name).alias(name) for name in stat_columns],
                F.count(F.when(F.col(stat_columns[0]) > 0, F.lit(1))).alias(f"{metric}_neighbor_count"),
                F.sum(F.col(observed_col).cast("long")).alias(f"{metric}_observed_neighbor_count"),
                F.sum(F.col(filled_col).cast("long")).alias(f"{metric}_filled_neighbor_count"),
            ])

        group_cols = ["_target_id", "target_game", "own_status", "month_index"]
        performance_by_month_df = (
            neighbor_aligned_stats_df.groupBy(*group_cols).agg(*filled_aggregations)
        )
        for metric in performance:
            weight_sum = F.col(f"_{metric}_weight_sum")
            mean_col = f"{metric}_mean"
            std_col = f"{metric}_std"
            # A metric's denominator includes only rows where that metric is present.
            performance_by_month_df = (
                performance_by_month_df
                .withColumn(mean_col, F.when(weight_sum > 0, F.col(f"_{metric}_weighted_sum") / weight_sum))
                .withColumn(
                    std_col,
                    F.when(
                        weight_sum > 0,
                        F.sqrt(F.greatest(
                            F.col(f"_{metric}_weighted_sq_sum") / weight_sum - F.pow(F.col(mean_col), 2),
                            F.lit(0.0),
                        )),
                    ),
                )
                .withColumn(f"{metric}_lower_2std", F.col(mean_col) - 2 * F.col(std_col))
                .withColumn(f"{metric}_higher_2std", F.col(mean_col) + 2 * F.col(std_col))
            )

        target_months_df = (
            performance_targets_df
            .withColumn("month_index", F.explode(F.sequence(F.lit(1), F.lit(FORECAST_MONTHS))))
            .withColumn("_forecast_month", F.add_months("curve_start_month", F.col("month_index") - 1))
            .withColumn("forecast_yearmonth", F.date_format("_forecast_month", "yyyyMM"))
        )
        performance_forecast_df = target_months_df.join(
            performance_by_month_df, on=group_cols, how="left"
        )
        for metric in performance:
            performance_forecast_df = performance_forecast_df.fillna({
                f"{metric}_neighbor_count": 0,
                f"{metric}_observed_neighbor_count": 0,
                f"{metric}_filled_neighbor_count": 0,
            })

        # Target actuals retain the original policy: slot-weight each raw source row.
        # Their metric-specific missing values do not enter that metric's denominator.
        actual_perf_aggs = []
        for metric in performance:
            valid_weight = F.when(F.col(metric).isNotNull(), F.col("no_of_slots"))
            actual_perf_aggs.append(
                F.when(
                    F.sum(valid_weight) > 0,
                    F.sum(valid_weight * F.col(metric)) / F.sum(valid_weight),
                ).alias(f"{metric}_actual")
            )
        actual_perf_monthly_df = (
            performance_source_df.groupBy("_game_key", "own_status", "_perf_month")
            .agg(*actual_perf_aggs)
            .withColumnRenamed("_perf_month", "_forecast_month")
        )
        performance_forecast_with_actuals_df = performance_forecast_df.join(
            actual_perf_monthly_df, on=["_game_key", "own_status", "_forecast_month"], how="left"
        )
        actual_perf_fill_window = (
            Window.partitionBy("_target_id", "own_status").orderBy("month_index")
            .rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
        for metric in performance:
            actual_col = f"{metric}_actual"
            filled_col = f"{metric}_actual_filled"
            performance_forecast_with_actuals_df = (
                performance_forecast_with_actuals_df
                .withColumn(
                    filled_col,
                    F.when(
                        F.col("_forecast_month") <= F.col("_target_perf_last_month"),
                        F.last(F.col(actual_col), ignorenulls=True).over(actual_perf_fill_window),
                    ),
                )
                .withColumn(
                    f"{metric}_actual_is_filled",
                    F.col(actual_col).isNull() & F.col(filled_col).isNotNull(),
                )
            )
        performance_forecast_with_actuals_df = (
            performance_forecast_with_actuals_df
            .drop("_game_key", "_target_start_month", "_target_perf_last_month", "_forecast_month")
        )
        # Keep the intermediate forecast DataFrame free of internal calendar join keys.
        performance_forecast_df = performance_forecast_df.drop(
            "_game_key", "_target_start_month", "_target_perf_last_month", "_forecast_month"
        )
        PERFORMANCE_RUN_ID = FORECAST_RUN_ID
        display(performance_forecast_with_actuals_df)

        # Freeze the outputs to be saved, and retain per-source selection/coverage history.
        assert_forecast_settings()
        if any(run_id != FORECAST_RUN_ID for run_id in [
            NEIGHBOR_SELECTION_RUN_ID, COMMERCIAL_RUN_ID, PERFORMANCE_RUN_ID
        ]):
            raise ValueError("A previous run's DataFrame is still in memory. Rerun selection and both forecasting cells.")

        audit_keys = ["_target_id", "neighbor_type", "own_status", "_neighbor_source_key"]
        commercial_availability_df = (
            neighbor_demand_curves_df
            .withColumn("_neighbor_source_key", normalized_name(F.col("matched_source_name")))
            .groupBy(*audit_keys)
            .agg(
                F.min("curve_start_month").alias("first_observed_source_month"),
                F.max("last_observed_source_month").alias("last_observed_source_month"),
                F.max("source_months_observed").cast("long").alias("source_months_observed"),
                F.countDistinct("month_index").alias("curve_months_available"),
            )
        )
        performance_availability_df = (
            neighbor_perf_df
            .withColumn("neighbor_type", F.lit("eilers"))
            .withColumn("_neighbor_source_key", normalized_name(F.col("matched_source_name")))
            .groupBy(*audit_keys)
            .agg(
                F.min("_perf_month").alias("first_observed_source_month"),
                F.max("_perf_month").alias("last_observed_source_month"),
                F.countDistinct("_perf_month").alias("source_months_observed"),
                F.least(F.max("month_index"), F.lit(FORECAST_MONTHS)).cast("long").alias("curve_months_available"),
            )
        )
        availability_df = commercial_availability_df.unionByName(performance_availability_df)

        # Eilers lists are retained once for each target population (owned/leased).
        neighbor_channel_metadata_df = target_channel_info_df.select("_target_id", "revenue_type", "own_status")
        forecast_neighbors_kept_df = (
            all_neighbors_kept_df.drop("own_status")
            .join(neighbor_channel_metadata_df, "_target_id", "inner")
            .where((F.col("neighbor_type") == "eilers")
                   | ((F.col("neighbor_type") == "sales") & (F.col("revenue_type") == "sales"))
                   | ((F.col("neighbor_type") == "rental") & (F.col("revenue_type") == "lease")))
            .join(availability_df, audit_keys, "left")
            .withColumn("has_asof_source_data", F.col("source_months_observed").isNotNull())
            .fillna({"source_months_observed": 0, "curve_months_available": 0})
            .withColumn("distance_cutoff", F.lit(float(DISTANCE_CUTOFF)))
            .withColumn("minimum_neighbors", F.lit(int(MIN_NEIGHBORS)))
            .withColumn("forecast_months", F.lit(int(FORECAST_MONTHS)))
        )

        # Check actual neighbor inputs; future target actuals are intentionally allowed.
        if neighbor_monthly_asof_df.where(
            F.col("_calendar_month") >= F.trunc("prediction_date", "month")
        ).limit(1).count():
            raise RuntimeError("A commercial neighbor observation violates the prediction cutoff.")
        if neighbor_perf_df.where(
            F.col("_perf_month") >= F.trunc("prediction_date", "month")
        ).limit(1).count():
            raise RuntimeError("An Eilers neighbor observation violates the prediction cutoff.")

        assert_unique(demand_forecast_curve_df, ["forecast_run_id", "target_game_key", "revenue_type", "month_index"], "Demand output")
        assert_unique(performance_forecast_with_actuals_df, ["forecast_run_id", "target_game_key", "own_status", "month_index"], "Performance output")
        assert_unique(forecast_neighbors_kept_df, ["forecast_run_id", "target_game_key", "neighbor_type", "own_status", "_neighbor_source_key"], "Retained neighbors")


        def dataset_signature(frame):
            """Order-independent multiset hash; no raw rows collected to pandas/driver."""
            columns = []
            for field in sorted(frame.schema.fields, key=lambda field: field.name):
                if field.name == "forecast_mlflow_run_id":
                    continue
                value = F.col(field.name)
                if isinstance(field.dataType, MapType):
                    value = F.map_from_entries(F.sort_array(F.map_entries(value)))
                columns.append(value.alias(field.name))
            row_hashes = frame.select(F.sha2(F.to_json(F.struct(*columns), {"ignoreNullFields": "false"}), 256).alias("_row_hash"))
            row = row_hashes.agg(
                F.count("_row_hash").alias("row_count"),
                F.sha2(F.concat_ws("|", F.sort_array(F.collect_list("_row_hash"))), 256).alias("sha256"),
            ).first()
            return {"row_count": int(row["row_count"]), "sha256": row["sha256"]}


        # Cache final outputs once; the final save cell validates these same outputs.
        OUTPUT_FRAMES = {
            "demand": demand_forecast_curve_df.cache(),
            "performance": performance_forecast_with_actuals_df.cache(),
            "neighbors": forecast_neighbors_kept_df.cache(),
        }
        OUTPUT_TABLES = {"demand": DEMAND_TABLE, "performance": PERFORMANCE_TABLE, "neighbors": NEIGHBORS_TABLE}
        OUTPUT_SIGNATURES = {label: dataset_signature(frame) for label, frame in OUTPUT_FRAMES.items()}
        PREPARED_SAVE_RUN_ID = FORECAST_RUN_ID
        PREPARED_SAVE_CONFIG_HASH = FORECAST_CONFIG_HASH

        expected_curve_rows = target_channel_info_df.count() * FORECAST_MONTHS
        for label in ["demand", "performance"]:
            if OUTPUT_SIGNATURES[label]["row_count"] != expected_curve_rows:
                raise RuntimeError(f"{label} lost target-month rows: expected {expected_curve_rows}, got {OUTPUT_SIGNATURES[label]['row_count']}.")
        print("Ready to save:", OUTPUT_SIGNATURES)
        display(demand_forecast_curve_df.groupBy("revenue_type").agg(
            F.count("forecast_value").alias("forecast_months"),
            F.sum(F.col("actual_is_observed").cast("long")).alias("observed_actual_months"),
            F.countDistinct(F.when(F.col("forecast_value").isNotNull(), F.col("target_game_key"))).alias("targets_with_forecasts"),
        ))

        # Save this run's three result tables, then publish its completion manifest.
        # The same run ID can be retried after a partial write without duplicating rows.
        import platform
        import time
        import mlflow
        from mlflow.tracking import MlflowClient
        from mlflow.entities import Param, Metric

        assert_forecast_settings()
        if PREPARED_SAVE_RUN_ID != FORECAST_RUN_ID or PREPARED_SAVE_CONFIG_HASH != FORECAST_CONFIG_HASH:
            raise ValueError("Outputs belong to a different setup. Rerun both forecasts and output preparation.")
        current_signatures = {label: dataset_signature(frame) for label, frame in OUTPUT_FRAMES.items()}
        if current_signatures != OUTPUT_SIGNATURES:
            raise ValueError("Prepared output content changed. Rerun setup and downstream cells for a fresh run.")
        save_identity = (FORECAST_RUN_ID, FORECAST_CONFIG_HASH, canonical_json(OUTPUT_SIGNATURES))
        if SAVE_STATE is not None and SAVE_STATE != save_identity:
            raise ValueError("This forecast ID already belongs to different output content.")


        def stored_run_frame(table):
            if not spark.catalog.tableExists(table):
                return None
            frame = spark.table(table)
            # The pre-existing 06 tables do not yet have forecast_run_id.
            if "forecast_run_id" not in frame.columns:
                return None
            return frame.where(F.col("forecast_run_id") == FORECAST_RUN_ID)


        def persisted_mlflow_ids(frame):
            if frame is None:
                return []
            if "forecast_mlflow_run_id" not in frame.columns:
                if frame.limit(1).count():
                    raise RuntimeError("This forecast ID exists without an MLflow identity.")
                return []
            return [row["forecast_mlflow_run_id"] for row in
                    frame.select("forecast_mlflow_run_id").distinct().limit(2).collect()]


        known_ids = set()
        for table in [*OUTPUT_TABLES.values(), RUNS_TABLE]:
            known_ids.update(persisted_mlflow_ids(stored_run_frame(table)))
        if None in known_ids or len(known_ids) > 1:
            raise RuntimeError("Conflicting saved MLflow identities for this forecast ID.")
        if known_ids:
            saved_run_id = next(iter(known_ids))
            if FORECAST_MLFLOW_RUN_ID is not None and FORECAST_MLFLOW_RUN_ID != saved_run_id:
                raise RuntimeError("A concurrent save owns this forecast ID. Reload it before retrying.")
            FORECAST_MLFLOW_RUN_ID = saved_run_id

        client = MlflowClient()
        experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT_PATH)
        if experiment is None:
            try:
                mlflow_experiment_id = client.create_experiment(MLFLOW_EXPERIMENT_PATH)
            except Exception:
                experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT_PATH)
                if experiment is None:
                    raise
                mlflow_experiment_id = experiment.experiment_id
        else:
            mlflow_experiment_id = experiment.experiment_id

        if FORECAST_MLFLOW_RUN_ID is None:
            run = client.create_run(mlflow_experiment_id, tags={
                "mlflow.runName": FORECAST_NAME.strip() or f"forecast_{SELECTED_SIMILARITY_EXPERIMENT_ID[-8:]}_{FORECAST_RUN_ID[-8:]}",
                "pipeline_stage": "forecast",
                "forecast_run_id": FORECAST_RUN_ID,
                "forecast_config_hash": FORECAST_CONFIG_HASH,
                "similarity_experiment_id": SELECTED_SIMILARITY_EXPERIMENT_ID,
                "upstream_similarity_mlflow_run_id": SIMILARITY_IDENTITY["similarity_mlflow_run_id"],
            })
            FORECAST_MLFLOW_RUN_ID = run.info.run_id
        run_id = FORECAST_MLFLOW_RUN_ID
        SAVE_STATE = save_identity


        def append_verified_output(label, frame):
            """Append immutable output once, and verify rows/hash/tracking identity."""
            table = OUTPUT_TABLES[label]
            expected = OUTPUT_SIGNATURES[label]
            existing = stored_run_frame(table)
            existing_count = 0 if existing is None else existing.count()
            if existing_count:
                if dataset_signature(existing.select(*frame.columns)) != expected or persisted_mlflow_ids(existing) != [run_id]:
                    raise RuntimeError(f"Different/incomplete {label} content already exists for this forecast ID.")
            else:
                (frame.withColumn("forecast_mlflow_run_id", F.lit(run_id))
                 .write.format("delta").mode("append").option("mergeSchema", "true")
                 .option("txnAppId", f"forecast:{FORECAST_RUN_ID}:{label}").option("txnVersion", 0)
                 .option("userMetadata", json.dumps({"forecast_run_id": FORECAST_RUN_ID, "output": label, "mlflow_run_id": run_id}))
                 .saveAsTable(table))

            persisted = stored_run_frame(table)
            if persisted is None:
                raise RuntimeError(f"{label} output table was not created.")
            if dataset_signature(persisted.select(*frame.columns)) != expected:
                raise RuntimeError(f"{label} row count or content hash differs after saving.")
            if expected["row_count"] and persisted_mlflow_ids(persisted) != [run_id]:
                raise RuntimeError(f"{label} was written by a different MLflow run.")
            commit = (
                spark.sql(f"DESCRIBE HISTORY {quoted_table_name(table)}")
                .where(F.get_json_object("userMetadata", "$.forecast_run_id") == FORECAST_RUN_ID)
                .agg(F.min("version").alias("v")).first()["v"]
            )
            return {"table": table, "delta_commit_version": None if commit is None else int(commit), **expected}


        try:
            params = {
                "similarity_experiment_id": SELECTED_SIMILARITY_EXPERIMENT_ID,
                "similarity_config_hash": SIMILARITY_IDENTITY["similarity_config_hash"],
                "distance_method": FORECAST_CONFIG["distance_method"],
                "parent_game_process": FORECAST_CONFIG["parent_game_process"],
                "distance_cutoff": DISTANCE_CUTOFF, "minimum_neighbors": MIN_NEIGHBORS,
                "forecast_months": FORECAST_MONTHS,
                "performance_slot_counts": canonical_json(FORECAST_CONFIG["slot_counts"]),
                "lease_value_column": LEASE_VALUE_COLUMN,
                "prediction_date_rule": "game_release_date",
                "neighbor_observation_rule": "strictly_before_prediction_month",
                "implementation_version": IMPLEMENTATION_VERSION,
            }
            params.update({f"weight_{group}": weight for group, weight in FORECAST_CONFIG["feature_group_weights"].items()})
            params.update({f"source_version_{label}": version for label, version in RESOLVED_SOURCE_VERSIONS.items()})
            metrics = {f"{label}_saved_rows": signature["row_count"] for label, signature in OUTPUT_SIGNATURES.items()}
            for row in OUTPUT_FRAMES["demand"].groupBy("revenue_type").agg(
                F.count("forecast_value").alias("predicted_months"),
                F.count("month_index").alias("target_months"),
                F.sum(F.col("actual_is_observed").cast("long")).alias("observed_actual_months"),
            ).collect():
                channel = row["revenue_type"]
                metrics[f"{channel}_predicted_months"] = row["predicted_months"]
                metrics[f"{channel}_forecast_coverage"] = row["predicted_months"] / row["target_months"]
                metrics[f"{channel}_observed_actual_months"] = row["observed_actual_months"] or 0
            perf_metrics = []
            for metric in performance:
                perf_metrics.append(F.count(f"{metric}_mean").alias(f"{metric}_predicted_months"))
            perf_counts = OUTPUT_FRAMES["performance"].agg(*perf_metrics).first().asDict()
            metrics.update(perf_counts)
            client.log_batch(
                run_id, params=[Param(key, str(value)) for key, value in params.items()],
                metrics=[Metric(key, float(value), int(time.time() * 1000), 0) for key, value in metrics.items()],
                synchronous=True,
            )
            client.log_dict(run_id, FORECAST_CONFIG, "configuration.json")
            client.log_dict(run_id, {
                "python": platform.python_version(), "spark": spark.version,
                "mlflow": mlflow.__version__, "spark_session_timezone": spark.conf.get("spark.sql.session.timeZone"),
                "implementation_version": IMPLEMENTATION_VERSION,
            }, "environment.json")

            saved_outputs = {label: append_verified_output(label, frame) for label, frame in OUTPUT_FRAMES.items()}
            client.log_dict(run_id, {"forecast_run_id": FORECAST_RUN_ID, "outputs": saved_outputs}, "results/delta_outputs.json")

            # The manifest is the completion boundary for downstream readers.
            manifest_data = {
                "forecast_run_id": FORECAST_RUN_ID, "forecast_mlflow_run_id": run_id,
                "forecast_config_hash": FORECAST_CONFIG_HASH, "configuration_json": FORECAST_CONFIG_JSON,
                "experiment_id": SELECTED_SIMILARITY_EXPERIMENT_ID,
                "similarity_mlflow_run_id": SIMILARITY_IDENTITY["similarity_mlflow_run_id"],
                "similarity_config_hash": SIMILARITY_IDENTITY["similarity_config_hash"],
                "status": "complete", "started_at_utc": FORECAST_RUN_STARTED_AT_UTC,
                "source_versions_json": canonical_json(RESOLVED_SOURCE_VERSIONS),
                "output_signatures_json": canonical_json(OUTPUT_SIGNATURES),
                "output_locations_json": canonical_json(saved_outputs),
            }
            existing_manifest = stored_run_frame(RUNS_TABLE)
            manifests = [] if existing_manifest is None else existing_manifest.limit(2).collect()
            if not manifests:
                manifest_df = spark.createDataFrame([manifest_data])
                (manifest_df.write.format("delta").mode("append").option("mergeSchema", "true")
                 .option("txnAppId", f"forecast:{FORECAST_RUN_ID}:manifest").option("txnVersion", 0)
                 .option("userMetadata", json.dumps({"forecast_run_id": FORECAST_RUN_ID, "output": "manifest"}))
                 .saveAsTable(RUNS_TABLE))
            verified_manifest = stored_run_frame(RUNS_TABLE).limit(2).collect()
            if len(verified_manifest) != 1 or any(
                verified_manifest[0][name] != manifest_data[name]
                for name in ["forecast_mlflow_run_id", "forecast_config_hash", "configuration_json", "output_signatures_json", "status"]
            ):
                raise RuntimeError("The saved completion manifest differs from this run.")
            client.set_tag(run_id, "forecast_save_status", "complete")
            client.set_terminated(run_id, status="FINISHED")
        except BaseException as error:
            try:
                client.set_tag(run_id, "forecast_save_status", "failed")
                client.set_tag(run_id, "failure_type", type(error).__name__)
                client.set_terminated(run_id, status="KILLED" if isinstance(error, KeyboardInterrupt) else "FAILED")
            except Exception:
                pass
            raise

        print(f"Saved forecast run: {FORECAST_RUN_ID}")
        print(f"MLflow run: {run_id}")
        print(f"Upstream similarity experiment: {SELECTED_SIMILARITY_EXPERIMENT_ID}")
        print(f"Completion manifest: {RUNS_TABLE}")
        print("Completed this experiment. Notebook 07 can load its saved forecast_run_id.")
        return {
            "experiment_id": SELECTED_SIMILARITY_EXPERIMENT_ID,
            "experiment_name": like_games_df.select("experiment_readable_name").first()["experiment_readable_name"] or SELECTED_SIMILARITY_EXPERIMENT_ID,
            "forecast_run_id": FORECAST_RUN_ID,
            "forecast_mlflow_run_id": FORECAST_MLFLOW_RUN_ID,
            "status": "complete",
            **{f"{label}_rows": signature["row_count"] for label, signature in OUTPUT_SIGNATURES.items()},
            **{key: value for key, value in metrics.items() if key.endswith("predicted_months") or key.endswith("forecast_coverage")},
        }
    finally:
        # Retain IDs for safe retries of partial writes in this Python session.
        run_state["forecast_mlflow_run_id"] = FORECAST_MLFLOW_RUN_ID
        run_state["save_identity"] = SAVE_STATE
        for frame in OUTPUT_FRAMES.values():
            frame.unpersist()


In [0]:
# Run ALL selected experiments; retry this cell to resume a partial batch.
if public_forecast_settings() != LOADED_SETTINGS:
    raise ValueError("Settings changed. Rerun the notebook from loading before executing the batch.")
batch_key = canonical_json({
    "settings": LOADED_SETTINGS, "source_versions": RESOLVED_SOURCE_VERSIONS,
    "experiments": SELECTED_SIMILARITY_EXPERIMENT_IDS,
})
if globals().get("FORECAST_BATCH_STATE", {}).get("key") != batch_key:
    FORECAST_BATCH_STATE = {"key": batch_key, "experiments": {}}

for position, experiment_id in enumerate(SELECTED_SIMILARITY_EXPERIMENT_IDS, 1):
    state = FORECAST_BATCH_STATE["experiments"].setdefault(experiment_id, {})
    if state.get("result", {}).get("status") == "complete":
        print(f"[{position}/{len(SELECTED_SIMILARITY_EXPERIMENT_IDS)}] Already completed: {experiment_id}")
        continue
    print(f"[{position}/{len(SELECTED_SIMILARITY_EXPERIMENT_IDS)}] Forecasting: {experiment_id}")
    try:
        state["result"] = run_similarity_forecast(experiment_id, state)
    except Exception as error:
        state["result"] = {
            "experiment_id": experiment_id, "forecast_run_id": state.get("forecast_run_id"),
            "status": "failed", "error": f"{type(error).__name__}: {error}",
        }
        print(f"FAILED {experiment_id}: {type(error).__name__}: {error}")

# Show every outcome, including failures and zero forecast coverage.
import pandas as pd
forecast_batch_summary_pdf = pd.DataFrame([
    FORECAST_BATCH_STATE["experiments"][experiment_id]["result"]
    for experiment_id in SELECTED_SIMILARITY_EXPERIMENT_IDS
])
display(forecast_batch_summary_pdf)
failed_ids = forecast_batch_summary_pdf.loc[forecast_batch_summary_pdf["status"].ne("complete"), "experiment_id"].tolist()
if failed_ids:
    raise RuntimeError(f"Forecast batch has failed experiments: {failed_ids}. Review the errors above, then rerun this cell to retry them; completed experiments are skipped.")
print(f"Saved demand and performance outputs for all {len(SELECTED_SIMILARITY_EXPERIMENT_IDS)} selected experiments.")
print("Next: rerun notebook 07 with FORECAST_RUN_IDS = [] and COHORT_ID = '' to compare these compatible runs.")
